In [1]:
import pandas as pd

In [2]:
# Chargement des trois fichiers sources
dvf = pd.read_excel('/Users/bader/Desktop/Data/OpenClassrooms (1)/P5/Données-immo/Valeurs-foncières.xlsx')
geo = pd.read_excel('/Users/bader/Desktop/Data/OpenClassrooms (1)/P5/Données-immo/fr-esr-referentiel-geographique.xlsx')
insee = pd.read_excel('/Users/bader/Desktop/Data/OpenClassrooms (1)/P5/Données-immo/donnees_communes.xlsx')

In [4]:
# Aperçu des colonnes disponibles dans chaque fichier source
print("=== DVF ===")
print(dvf.columns.tolist())

print("\n=== GEO ===")
print(geo.columns.tolist())

print("\n=== INSEE ===")
print(insee.columns.tolist())

=== DVF ===
['Code service CH', 'Reference document', '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI', 'No disposition', 'Date mutation', 'Nature mutation', 'Valeur fonciere', 'No voie', 'B/T/Q', 'Code type de voie', 'Type de voie', 'Code voie', 'Voie', 'Code ID commune', 'Code postal', 'Commune', 'Code departement', 'Code commune', 'Préfixe de section', 'Section', 'No plan', 'No Volume', '1er lot', 'Surface Carrez du 1er lot', '2eme lot', 'Surface Carrez du 2eme lot', '3eme lot', 'Surface Carrez du 3eme lot', '4eme lot', 'Surface Carrez du 4eme lot', '5eme lot', 'Surface Carrez du 5eme lot', 'Nombre de lots', 'Code type local', 'Type local', 'Identifiant local', 'Surface reelle bati', 'Nombre pieces principales', 'Nature culture', 'Nature culture speciale', 'Surface terrain', "Nom de l'acquereur"]

=== GEO ===
['regrgp_nom', 'reg_nom', 'reg_nom_old', 'aca_nom', 'dep_nom', 'com_code', 'com_code1', 'com_code2', 'com_id', 'com_nom_maj_court', 'com

In [6]:
# ── TABLE REGION ──────────────────────────────────────────────────────────────
# Sélection des colonnes nécessaires depuis le fichier géographique
region = geo[['reg_code', 'reg_nom']].copy()

# Suppression des doublons (une ligne par région)
region = region.drop_duplicates()

# Vérification
print(f"Nombre de régions : {len(region)}")
display(region.head())

Nombre de régions : 19


,reg_code,reg_nom
0,84,Auvergne-Rhône-Alpes
459,32,Hauts-de-France
1614,93,Provence-Alpes-Côte d'Azur
2555,44,Grand Est
3058,76,Occitanie


In [8]:
# ── TABLE REGION ──────────────────────────────────────────────────────────────
# Sélection des colonnes nécessaires depuis le fichier géographique
region = geo[['reg_code', 'reg_nom']].copy()

# Suppression des doublons (une ligne par région)
region = region.drop_duplicates()

# Vérification du résultat complet
print(f"Nombre de régions : {len(region)}")
pd.set_option('display.max_rows', None)
display(region)

Nombre de régions : 19


,reg_code,reg_nom
0,84,Auvergne-Rhône-Alpes
459,32,Hauts-de-France
1614,93,Provence-Alpes-Côte d'Azur
2555,44,Grand Est
3058,76,Occitanie
4728,28,Normandie
5761,75,Nouvelle-Aquitaine
6672,24,Centre-Val de Loire
7251,27,Bourgogne-Franche-Comté
7968,53,Bretagne


In [10]:
# ── TABLE DEPARTEMENT ─────────────────────────────────────────────────────────
# Sélection des colonnes nécessaires depuis le fichier géographique
departement = geo[['dep_code', 'dep_nom', 'reg_code']].copy()

# Suppression des doublons (une ligne par département)
departement = departement.drop_duplicates()

# Vérification
print(f"Nombre de départements : {len(departement)}")
display(departement)

Nombre de départements : 109


,dep_code,dep_nom,reg_code
0,1,Ain,84
459,2,Aisne,32
1293,3,Allier,84
1614,4,Alpes-de-Haute-Provence,93
1859,5,Hautes-Alpes,93
2043,6,Alpes-Maritimes,93
2206,7,Ardèche,84
2555,8,Ardennes,44
3058,9,Ariège,76
3400,10,Aube,44


In [12]:
# Vérification des valeurs
print(geo['com_code'].head())
print(insee[['CODDEP', 'CODCOM']].head())

0    01001
1    01002
2    01003
3    01004
4    01005
Name: com_code, dtype: object
  CODDEP  CODCOM
0     01       1
1     01       2
2     01       4
3     01       5
4     01       6


In [14]:
# Création de la clé de jointure dans INSEE
# CODCOM doit être sur 3 chiffres (zfill) pour correspondre à com_code dans GEO
insee['id_codedep_codecommune'] = (
    insee['CODDEP'].astype(str) + 
    insee['CODCOM'].astype(str).str.zfill(3)
)

# Vérification
print(insee[['CODDEP', 'CODCOM', 'id_codedep_codecommune']].head())

  CODDEP  CODCOM id_codedep_codecommune
0     01       1                  01001
1     01       2                  01002
2     01       4                  01004
3     01       5                  01005
4     01       6                  01006


In [16]:
# Sélection des colonnes utiles depuis GEO
geo_commune = geo[['com_code', 'dep_code', 'com_nom', 'com_code1']].copy()
geo_commune = geo_commune.rename(columns={
    'com_code': 'id_codedep_codecommune',
    'com_nom': 'nom_commune',
    'com_code1': 'code_postal'
})

# Sélection des colonnes utiles depuis INSEE
insee_commune = insee[['id_codedep_codecommune', 'PMUN', 'PTOT']].copy()
insee_commune = insee_commune.rename(columns={
    'PMUN': 'pmun',
    'PTOT': 'ptot'
})

# Jointure sur la clé commune
commune = geo_commune.merge(insee_commune, on='id_codedep_codecommune', how='left')

# Vérification
print(f"Nombre de communes : {len(commune)}")
display(commune.head())

Nombre de communes : 38916


,id_codedep_codecommune,dep_code,nom_commune,code_postal,pmun,ptot
0,01001,1,L'Abergement-Clémenciat,1001,779.0,798.0
1,01002,1,L'Abergement-de-Varey,1002,256.0,257.0
2,01003,1,Amareins,1003,NaN,NaN
3,01004,1,Ambérieu-en-Bugey,1004,14134.0,14514.0
4,01005,1,Ambérieux-en-Dombes,1005,1751.0,1776.0


In [18]:
# Vérification du code postal
display(commune[['id_codedep_codecommune', 'nom_commune', 'code_postal', 'pmun', 'ptot']].head(10))

,id_codedep_codecommune,nom_commune,code_postal,pmun,ptot
0,01001,L'Abergement-Clémenciat,1001,779.0,798.0
1,01002,L'Abergement-de-Varey,1002,256.0,257.0
2,01003,Amareins,1003,NaN,NaN
3,01004,Ambérieu-en-Bugey,1004,14134.0,14514.0
4,01005,Ambérieux-en-Dombes,1005,1751.0,1776.0
5,01006,Ambléon,1006,112.0,118.0
6,01007,Ambronay,1007,2800.0,2915.0
7,01008,Ambutrix,1008,762.0,777.0
8,01009,Andert-et-Condon,1009,326.0,335.0
9,01010,Anglefort,1010,1105.0,1122.0


In [20]:
# Inspection des colonnes de GEO pour trouver le vrai code postal
print(geo[['com_code', 'com_code1', 'com_code2']].head(10))

  com_code com_code1 com_code2
0    01001      1001      1001
1    01002      1002      1002
2    01003      1003      1003
3    01004      1004      1004
4    01005      1005      1005
5    01006      1006      1006
6    01007      1007      1007
7    01008      1008      1008
8    01009      1009      1009
9    01010      1010      1010


In [22]:
# Vérification du code postal dans DVF
print(dvf[['Code postal', 'Commune', 'Code departement', 'Code commune']].head(10))

   Code postal               Commune Code departement  Code commune
0       1170.0                CHEVRY               01           103
1       6160.0               ANTIBES               06             4
2       6000.0                  NICE               06            88
3       6700.0  SAINT LAURENT DU VAR               06           123
4      13400.0               AUBAGNE               13             5
5      13600.0             LA CIOTAT               13            28
6      13008.0        MARSEILLE 8EME               13           208
7      13012.0       MARSEILLE 12EME               13           212
8      14510.0              HOULGATE               14           338
9      14100.0               LISIEUX               14           366


In [24]:
# Création de la clé de jointure dans DVF
dvf['id_codedep_codecommune'] = (
    dvf['Code departement'].astype(str) + 
    dvf['Code commune'].astype(str).str.zfill(3)
)

# Extraction code postal unique par commune
code_postal = dvf[['id_codedep_codecommune', 'Code postal']].drop_duplicates(subset='id_codedep_codecommune')
code_postal = code_postal.rename(columns={'Code postal': 'code_postal'})

print(code_postal.head(10))

  id_codedep_codecommune  code_postal
0                  01103       1170.0
1                  06004       6160.0
2                  06088       6000.0
3                  06123       6700.0
4                  13005      13400.0
5                  13028      13600.0
6                  13208      13008.0
7                  13212      13012.0
8                  14338      14510.0
9                  14366      14100.0


In [26]:
# Correction du code postal : entier puis zfill(5) pour avoir 5 chiffres
code_postal['code_postal'] = (
    code_postal['code_postal']
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

print(code_postal.head(10))

  id_codedep_codecommune code_postal
0                  01103       01170
1                  06004       06160
2                  06088       06000
3                  06123       06700
4                  13005       13400
5                  13028       13600
6                  13208       13008
7                  13212       13012
8                  14338       14510
9                  14366       14100


In [28]:
# Reconstruction de la table COMMUNE sans le mauvais code_postal
geo_commune = geo[['com_code', 'dep_code', 'com_nom']].copy()
geo_commune = geo_commune.rename(columns={
    'com_code': 'id_codedep_codecommune',
    'com_nom': 'nom_commune'
})

# Jointure avec INSEE pour pmun et ptot
commune = geo_commune.merge(insee_commune, on='id_codedep_codecommune', how='left')

# Jointure avec DVF pour le code postal
commune = commune.merge(code_postal, on='id_codedep_codecommune', how='left')

# Réorganisation des colonnes dans l'ordre du schéma
commune = commune[['id_codedep_codecommune', 'dep_code', 'code_postal', 'nom_commune', 'pmun', 'ptot']]

# Vérification
print(f"Nombre de communes : {len(commune)}")
display(commune.head(10))

Nombre de communes : 38916


,id_codedep_codecommune,dep_code,code_postal,nom_commune,pmun,ptot
0,01001,1,NaN,L'Abergement-Clémenciat,779.0,798.0
1,01002,1,NaN,L'Abergement-de-Varey,256.0,257.0
2,01003,1,NaN,Amareins,NaN,NaN
3,01004,1,01500,Ambérieu-en-Bugey,14134.0,14514.0
4,01005,1,NaN,Ambérieux-en-Dombes,1751.0,1776.0
5,01006,1,NaN,Ambléon,112.0,118.0
6,01007,1,NaN,Ambronay,2800.0,2915.0
7,01008,1,NaN,Ambutrix,762.0,777.0
8,01009,1,NaN,Andert-et-Condon,326.0,335.0
9,01010,1,NaN,Anglefort,1105.0,1122.0


In [30]:
# Correction dep_code : zfill(2) pour garder le format 01, 06, 75...
commune['dep_code'] = commune['dep_code'].astype(str).str.zfill(2)

# Vérification
display(commune.head(10))

,id_codedep_codecommune,dep_code,code_postal,nom_commune,pmun,ptot
0,01001,01,NaN,L'Abergement-Clémenciat,779.0,798.0
1,01002,01,NaN,L'Abergement-de-Varey,256.0,257.0
2,01003,01,NaN,Amareins,NaN,NaN
3,01004,01,01500,Ambérieu-en-Bugey,14134.0,14514.0
4,01005,01,NaN,Ambérieux-en-Dombes,1751.0,1776.0
5,01006,01,NaN,Ambléon,112.0,118.0
6,01007,01,NaN,Ambronay,2800.0,2915.0
7,01008,01,NaN,Ambutrix,762.0,777.0
8,01009,01,NaN,Andert-et-Condon,326.0,335.0
9,01010,01,NaN,Anglefort,1105.0,1122.0


In [32]:
# ── TABLE BIEN ────────────────────────────────────────────────────────────────
# Sélection des colonnes nécessaires depuis DVF
bien = dvf[[
    'No voie',
    'B/T/Q',
    'Type de voie',
    'Voie',
    'Type local',
    'Surface reelle bati',
    'Nombre pieces principales',
    'Surface terrain',
    'id_codedep_codecommune'
]].copy()

# Renommage des colonnes selon le schéma
bien = bien.rename(columns={
    'No voie': 'no_voie',
    'B/T/Q': 'btq',
    'Type de voie': 'type_voie',
    'Voie': 'voie',
    'Type local': 'type_local',
    'Surface reelle bati': 'surface_reelle_bati',
    'Nombre pieces principales': 'nb_pieces_principales',
    'Surface terrain': 'surface_terrain'
})

# Création de l'id_bien autoincrement
bien = bien.reset_index(drop=True)
bien.insert(0, 'id_bien', bien.index + 1)

# Vérification
print(f"Nombre de biens : {len(bien)}")
display(bien.head())

Nombre de biens : 34169


,id_bien,no_voie,btq,type_voie,voie,type_local,surface_reelle_bati,nb_pieces_principales,surface_terrain,id_codedep_codecommune
0,1,347.0,NaN,RUE,DU CHATEAU,Appartement,48,3,NaN,01103
1,2,4.0,NaN,BD,EDOUARD BAUDOIN,Appartement,40,1,NaN,06004
2,3,20.0,B,RUE,MARCEAU,Appartement,82,3,NaN,06088
3,4,550.0,NaN,RTE,DES VESPINS RN7,Appartement,27,1,NaN,06123
4,5,9300.0,NaN,RES,LES ARPEGES BD DES ABA,Appartement,47,2,NaN,13005


In [34]:
# ── TABLE VENTE ───────────────────────────────────────────────────────────────
# Sélection des colonnes nécessaires depuis DVF
vente = dvf[[
    'Date mutation',
    'Nature mutation',
    'Valeur fonciere'
]].copy()

# Renommage des colonnes selon le schéma
vente = vente.rename(columns={
    'Date mutation': 'date_mutation',
    'Nature mutation': 'nature_mutation',
    'Valeur fonciere': 'valeur_fonciere'
})

# Création des IDs autoincrement
vente.insert(0, 'id_vente', range(1, len(vente) + 1))
vente.insert(1, 'id_bien', range(1, len(vente) + 1))

# Vérification
print(f"Nombre de ventes : {len(vente)}")
display(vente.head())

Nombre de ventes : 34169


,id_vente,id_bien,date_mutation,nature_mutation,valeur_fonciere
0,1,1,2020-01-02,Vente,165000.0
1,2,2,2020-01-02,Vente,355680.0
2,3,3,2020-01-02,Vente,229500.0
3,4,4,2020-01-02,Vente,125000.0
4,5,5,2020-01-02,Vente,90000.0


In [37]:
# ── EXPORT DES 5 CSV ──────────────────────────────────────────────────────────
import os

output_path = '/Users/bader/Desktop/Data/OpenClassrooms (1)/P5/csv/'

# Création du dossier si inexistant
os.makedirs(output_path, exist_ok=True)

region.to_csv(output_path + 'region.csv', index=False)
departement.to_csv(output_path + 'departement.csv', index=False)
commune.to_csv(output_path + 'commune.csv', index=False)
bien.to_csv(output_path + 'bien.csv', index=False)
vente.to_csv(output_path + 'vente.csv', index=False)

print("✅ region.csv exporté")
print("✅ departement.csv exporté")
print("✅ commune.csv exporté")
print("✅ bien.csv exporté")
print("✅ vente.csv exporté")

✅ region.csv exporté
✅ departement.csv exporté
✅ commune.csv exporté
✅ bien.csv exporté
✅ vente.csv exporté


In [39]:
# Conversion pmun et ptot en integer (supprimer les décimales)
# Les NaN sont conservés (communes sans données INSEE)
commune['pmun'] = commune['pmun'].astype('Int64')
commune['ptot'] = commune['ptot'].astype('Int64')

# Vérification
display(commune.head())

,id_codedep_codecommune,dep_code,code_postal,nom_commune,pmun,ptot
0,01001,01,NaN,L'Abergement-Clémenciat,779,798
1,01002,01,NaN,L'Abergement-de-Varey,256,257
2,01003,01,NaN,Amareins,<NA>,<NA>
3,01004,01,01500,Ambérieu-en-Bugey,14134,14514
4,01005,01,NaN,Ambérieux-en-Dombes,1751,1776


In [41]:
# Re-export commune corrigé
commune.to_csv(output_path + 'commune.csv', index=False)
print("✅ commune.csv re-exporté")

✅ commune.csv re-exporté


In [43]:
# Est-ce que les DOM sont présents dans le fichier DVF ?
print(dvf[dvf['Code departement'].astype(str).str.startswith('97')]['Code departement'].unique())

['972' '971' '973' '974']


In [45]:
print(departement['dep_code'].head(10))

0        1
459      2
1293     3
1614     4
1859     5
2043     6
2206     7
2555     8
3058     9
3400    10
Name: dep_code, dtype: object


In [47]:
# Correction dep_code dans departement : zfill(2)
departement['dep_code'] = departement['dep_code'].astype(str).str.zfill(2)

# Vérification
print(departement['dep_code'].head(10))

0       01
459     02
1293    03
1614    04
1859    05
2043    06
2206    07
2555    08
3058    09
3400    10
Name: dep_code, dtype: object


In [49]:
# Re-export departement corrigé
departement.to_csv(output_path + 'departement.csv', index=False)
print("✅ departement.csv re-exporté")

✅ departement.csv re-exporté


In [51]:
print(bien.columns.tolist())

['id_bien', 'no_voie', 'btq', 'type_voie', 'voie', 'type_local', 'surface_reelle_bati', 'nb_pieces_principales', 'surface_terrain', 'id_codedep_codecommune']


In [53]:
# Vérifier les premières lignes du CSV exporté
pd.read_csv(output_path + 'bien.csv').head(3)

,id_bien,no_voie,btq,type_voie,voie,type_local,surface_reelle_bati,nb_pieces_principales,surface_terrain,id_codedep_codecommune
0,1,347.0,NaN,RUE,DU CHATEAU,Appartement,48,3,NaN,01103
1,2,4.0,NaN,BD,EDOUARD BAUDOIN,Appartement,40,1,NaN,06004
2,3,20.0,B,RUE,MARCEAU,Appartement,82,3,NaN,06088


In [55]:
# Correction des colonnes numériques avec décimales
bien['no_voie'] = bien['no_voie'].astype('Int64')
bien['surface_reelle_bati'] = bien['surface_reelle_bati'].astype('Int64')
bien['nb_pieces_principales'] = bien['nb_pieces_principales'].astype('Int64')
bien['surface_terrain'] = bien['surface_terrain'].astype('Int64')

# Re-export
bien.to_csv(output_path + 'bien.csv', index=False)
print("✅ bien.csv re-exporté")

✅ bien.csv re-exporté


In [57]:
import sqlite3

# Connexion à la BDD SQLite (crée le fichier si inexistant)
conn = sqlite3.connect('/Users/bader/Desktop/Data/OpenClassrooms (1)/P5/immo_p5.db')

print("✅ Connexion SQLite établie")

✅ Connexion SQLite établie


In [61]:
# Chargement des 5 tables dans SQLite
region.to_sql('REGION', conn, if_exists='replace', index=False)
departement.to_sql('DEPARTEMENT', conn, if_exists='replace', index=False)
commune.to_sql('COMMUNE', conn, if_exists='replace', index=False)
bien.to_sql('BIEN', conn, if_exists='replace', index=False)
vente.to_sql('VENTE', conn, if_exists='replace', index=False)

conn.close()
print("✅ BDD SQLite chargée")

ProgrammingError: Cannot operate on a closed database.

In [63]:
# Reconnexion à la BDD SQLite
conn = sqlite3.connect('/Users/bader/Desktop/Data/OpenClassrooms (1)/P5/immo_p5.db')

# Insertion dans l'ordre des clés étrangères
region.to_sql('REGION', conn, if_exists='append', index=False)
departement.to_sql('DEPARTEMENT', conn, if_exists='append', index=False)
commune.to_sql('COMMUNE', conn, if_exists='append', index=False)
bien.to_sql('BIEN', conn, if_exists='append', index=False)
vente.to_sql('VENTE', conn, if_exists='append', index=False)

conn.close()
print("✅ Données réinsérées avec contraintes")

IntegrityError: UNIQUE constraint failed: REGION.reg_code

In [65]:
# Vérification du chargement
conn = sqlite3.connect('/Users/bader/Desktop/Data/OpenClassrooms (1)/P5/immo_p5.db')

for table in ['REGION', 'DEPARTEMENT', 'COMMUNE', 'BIEN', 'VENTE']:
    count = pd.read_sql(f"SELECT COUNT(*) as nb FROM {table}", conn).iloc[0]['nb']
    print(f"{table} : {count} lignes")

conn.close()

REGION : 19 lignes
DEPARTEMENT : 109 lignes
COMMUNE : 38916 lignes
BIEN : 34169 lignes
VENTE : 34169 lignes


In [66]:
# ── VERIFICATION DE L'INTEGRITE DES DONNEES ───────────────────────────────────

print("=== VERIFICATION DES TABLES ===\n")
print(f"REGION      : {len(region)} lignes (attendu : 19)")
print(f"DEPARTEMENT : {len(departement)} lignes (attendu : 109)")
print(f"COMMUNE     : {len(commune)} lignes (attendu : 38 916)")
print(f"BIEN        : {len(bien)} lignes (attendu : 34 169)")
print(f"VENTE       : {len(vente)} lignes (attendu : 34 169)")

print("\n=== VALEURS MANQUANTES ===\n")
print(f"COMMUNE - code_postal NaN : {commune['code_postal'].isna().sum()} communes")
print(f"  → Normal : DVF ne contient que les communes avec transactions immobilières")
print(f"COMMUNE - pmun NaN        : {commune['pmun'].isna().sum()} communes")
print(f"  → Normal : communes absentes du recensement INSEE")
print(f"VENTE - valeur_fonciere NaN : {vente['valeur_fonciere'].isna().sum()} ventes")
print(f"  → Transactions sans prix déclaré dans le fichier source DVF")

print("\n=== PERIODE COUVERTE ===\n")
print(f"Données DVF : 1er semestre 2020 (échantillon fourni par OpenClassrooms)")
print(f"Nature mutation unique : {vente['nature_mutation'].unique()}")

=== VERIFICATION DES TABLES ===

REGION      : 19 lignes (attendu : 19)
DEPARTEMENT : 109 lignes (attendu : 109)
COMMUNE     : 38916 lignes (attendu : 38 916)
BIEN        : 34169 lignes (attendu : 34 169)
VENTE       : 34169 lignes (attendu : 34 169)

=== VALEURS MANQUANTES ===

COMMUNE - code_postal NaN : 35791 communes
  → Normal : DVF ne contient que les communes avec transactions immobilières
COMMUNE - pmun NaN        : 3925 communes
  → Normal : communes absentes du recensement INSEE
VENTE - valeur_fonciere NaN : 18 ventes
  → Transactions sans prix déclaré dans le fichier source DVF

=== PERIODE COUVERTE ===

Données DVF : 1er semestre 2020 (échantillon fourni par OpenClassrooms)
Nature mutation unique : ['Vente']
